# Pilot solver runs on Colab

Fits the flow maxent solver over the committed LLM elicitation cache
(`metaculus/llm_cache_full.json`) — **no API keys needed**, the LLM half is
already done. Three arms, equal steps: `abs` (legacy absolute penalty),
`logit` (log-odds penalty, new default), `logit --robust` (gated
constraints).

Runtime guide: a few seconds/question on GPU, ~20-30 s on CPU — so
147 questions x 3 arms is well under an hour on a GPU runtime. To split
across k parallel sessions, give each one `--shard i/k` and its own
`--out` file, then merge with the last cell.

In [ ]:
!git clone https://github.com/amdson/calibrated_response.git
%cd calibrated_response
%pip install -q -e .

In [ ]:
# use the GPU when the runtime has one (the solver script defaults to CPU
# only when JAX_PLATFORMS is unset)
import os, subprocess
has_gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
os.environ["JAX_PLATFORMS"] = "cuda" if has_gpu else "cpu"
print("JAX_PLATFORMS =", os.environ["JAX_PLATFORMS"])

In [ ]:
# RESYNC: after a local `git commit -am wip && git push`, rerun this cell —
# no re-clone or reinstall needed (editable install picks up code changes).
# Untracked outputs (arm_*.json) survive the reset.
!git fetch origin && git reset --hard origin/main

In [ ]:
# arm 1: legacy absolute penalty (the pre-fix baseline)
# Resume-safe: rerunning a cell skips finished entries.
!python metaculus/run_flow_solver.py --prob-penalty abs --out metaculus/arm_abs.json

In [ ]:
# arm 2: logit-space penalty (new default)
!python metaculus/run_flow_solver.py --prob-penalty logit --out metaculus/arm_logit.json

In [ ]:
!python metaculus/pilot_diagnostics.py --predictions \
    metaculus/arm_abs.json metaculus/arm_logit.json metaculus/arm_logit_robust.json

In [ ]:
# get the results off the runtime (pick one)
from google.colab import files
for f in ("arm_abs", "arm_logit", "arm_logit_robust"):
    files.download(f"metaculus/{f}.json")
# ...or mount Drive and copy:
# from google.colab import drive; drive.mount('/content/drive')
# !cp metaculus/arm_*.json /content/drive/MyDrive/

In [ ]:
# OPTIONAL: merge shard outputs from parallel sessions into one file
# import json, glob
# merged = {}
# for p in sorted(glob.glob("metaculus/arm_logit_shard*.json")):
#     merged.update(json.load(open(p)))
# json.dump(merged, open("metaculus/arm_logit.json", "w"), indent=1)
# print(len(merged), "rows merged")

In [ ]:
# OPTIONAL: merge shard outputs from parallel sessions into one file
# import json, glob
# merged = {}
# for p in sorted(glob.glob("metaculus/flow_predictions_shard*.json")):
#     merged.update(json.load(open(p)))
# json.dump(merged, open("metaculus/flow_predictions.json", "w"), indent=1)
# print(len(merged), "rows merged")